In [1]:
# """
# Intensity → Temperature Map Conversion
# ----------------------------------------
# Uses the linear calibration  T = slope * I + intercept
# derived from the intensity–temperature regression.

# For every case folder under INFRARED_ROOT:
#   <case>/time_average/window_1_corrected_interp.csv
#   <case>/time_average/window_2_corrected_interp.csv

# Outputs are saved to:
#   <case>/time_average/average_temperature/window_1_temperature.csv
#   <case>/time_average/average_temperature/window_2_temperature.csv
# """

# import os
# import glob
# import pandas as pd
# import numpy as np

# # ── Calibration: T = SLOPE * I + INTERCEPT ──────────────────────────────────
# # Paste the values printed by intensity_temperature_regression.py here
# SLOPE     =  0.0251   # °C / count   ← replace with your regression slope
# INTERCEPT =  23.3     # °C           ← replace with your regression intercept

# # ── Root directory ────────────────────────────────────────────────────────────
# INFRARED_ROOT = r"D:\NRC\FCAI_combined\effusion_coupon\experiment\infared_data"

# # Number of header lines to skip in each CSV
# HEADER_LINES = 10

# # ── Helper ────────────────────────────────────────────────────────────────────
# def intensity_csv_to_temperature(src_path: str, dst_path: str) -> None:
#     """Read an intensity map CSV, apply calibration, save temperature map."""
#     intensity = pd.read_csv(src_path, header=None, skiprows=HEADER_LINES).values.astype(float)
#     temperature = SLOPE * intensity + INTERCEPT
#     pd.DataFrame(temperature).to_csv(dst_path, index=False, header=False,
#                                      float_format="%.4f")

# # ── Main loop ─────────────────────────────────────────────────────────────────
# case_dirs = [
#     d for d in glob.glob(os.path.join(INFRARED_ROOT, "cr_*"))
#     if os.path.isdir(d)
# ]

# if not case_dirs:
#     raise FileNotFoundError(f"No case folders found under:\n  {INFRARED_ROOT}\n"
#                             "Check that INFRARED_ROOT is correct.")

# print(f"Found {len(case_dirs)} case(s).\n")

# processed = 0
# skipped   = 0

# for case_dir in sorted(case_dirs):
#     case_name   = os.path.basename(case_dir)
#     time_avg    = os.path.join(case_dir, "time_average")
#     out_dir     = os.path.join(time_avg, "average_temperature")

#     win1_src = os.path.join(time_avg, "window_1_corrected_interp.csv")
#     win2_src = os.path.join(time_avg, "window_2_corrected_interp.csv")

#     # Skip if neither source file exists
#     if not os.path.isfile(win1_src) and not os.path.isfile(win2_src):
#         print(f"[SKIP]  {case_name}  — no source CSVs found in time_average/")
#         skipped += 1
#         continue

#     os.makedirs(out_dir, exist_ok=True)

#     for win_idx, src in enumerate([win1_src, win2_src], start=1):
#         if not os.path.isfile(src):
#             print(f"  [WARN] {case_name} — window_{win_idx} source not found, skipping")
#             continue
#         dst = os.path.join(out_dir, f"window_{win_idx}_temperature.csv")
#         intensity_csv_to_temperature(src, dst)
#         print(f"  [OK]   {case_name} → window_{win_idx}_temperature.csv")

#     processed += 1

# print(f"\nDone.  Processed: {processed}   Skipped: {skipped}")
# print(f"Temperature maps saved to each case's time_average/average_temperature/ folder.")

In [2]:
"""
Intensity → Temperature Map Conversion
----------------------------------------
Uses the linear calibration  T = slope * I + intercept
derived from the intensity–temperature regression.

For every case folder under INFRARED_ROOT:
  <case>/time_average/window_1_corrected_interp.csv
  <case>/time_average/window_2_corrected_interp.csv

Outputs saved to <case>/time_average/average_temperature/:
  window_1_temperature.csv
  window_2_temperature.csv
  window_1_temperature.png
  window_2_temperature.png
"""

import os
import glob
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe for batch processing
import matplotlib.pyplot as plt

# ── Calibration: T = SLOPE * I + INTERCEPT ──────────────────────────────────
# Paste the values printed by intensity_temperature_regression.py here
SLOPE     =  0.0154   # °C / count   ← replace with your regression slope
INTERCEPT =  72.5     # °C           ← replace with your regression intercept

# ── Root directory ────────────────────────────────────────────────────────────
INFRARED_ROOT = r"D:\FCAI\plain_coupon\infared_data"

# Number of header lines to skip in each CSV
HEADER_LINES = 10

# ── Plot settings ─────────────────────────────────────────────────────────────
COLORMAP        = "inferno"   # e.g. "jet", "hot", "plasma", "inferno"
FONT_SIZE_TITLE = 13
FONT_SIZE_LABEL = 11
FONT_SIZE_CBAR  = 10
FONT_SIZE_TICK  = 10
PLOT_DPI        = 150

plt.rcParams.update({
    "text.usetex":     True,
    "font.family":     "serif",
    "font.serif":      ["Computer Modern Roman"],
    "axes.titlesize":  FONT_SIZE_TITLE,
    "axes.labelsize":  FONT_SIZE_LABEL,
    "xtick.labelsize": FONT_SIZE_TICK,
    "ytick.labelsize": FONT_SIZE_TICK,
})

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_temperature(src_path):
    """Read intensity CSV and return calibrated temperature array."""
    intensity = pd.read_csv(src_path, header=None, skiprows=HEADER_LINES).values.astype(float)
    return SLOPE * intensity + INTERCEPT


def save_temperature_csv(temperature, dst_path):
    pd.DataFrame(temperature).to_csv(dst_path, index=False, header=False,
                                     float_format="%.4f")


def parse_case_title(case_name, win_idx):
    """Build a LaTeX-friendly title from folder name, e.g. cr_375_phi_09."""
    parts = case_name.split("_")
    try:
        cr_val  = parts[1]
        phi_raw = parts[3]
        if len(phi_raw) == 2:       # "09" → "0.9"
            phi_val = f"{int(phi_raw)/10:.1f}"
        elif len(phi_raw) == 3:     # "118" → "1.18"
            phi_val = f"{int(phi_raw)/100:.2f}"
        else:
            phi_val = phi_raw
        return (rf"CR = {cr_val} slpm,\ $\phi$ = {phi_val}"
                + "\n" + rf"Window {win_idx} --- Temperature Map")
    except (IndexError, ValueError):
        return rf"{case_name} --- Window {win_idx} Temperature Map"


def save_temperature_plot(temperature, dst_path, case_name, win_idx):
    """Save a square false-colour temperature-map plot."""
    fig, ax = plt.subplots(figsize=(6, 6))

    im = ax.imshow(temperature, cmap=COLORMAP, origin="upper", aspect="equal")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"Temperature ($^\circ$C)", fontsize=FONT_SIZE_CBAR)
    cbar.ax.tick_params(labelsize=FONT_SIZE_CBAR)

    ax.set_title(parse_case_title(case_name, win_idx))
    ax.set_xlabel(r"Pixel column")
    ax.set_ylabel(r"Pixel row")

    plt.tight_layout()
    fig.savefig(dst_path, dpi=PLOT_DPI)
    plt.close(fig)


# ── Main loop ─────────────────────────────────────────────────────────────────
case_dirs = [
    d for d in glob.glob(os.path.join(INFRARED_ROOT, "cr_*"))
    if os.path.isdir(d)
]

if not case_dirs:
    raise FileNotFoundError(
        f"No case folders found under:\n  {INFRARED_ROOT}\n"
        "Check that INFRARED_ROOT is correct."
    )

print(f"Found {len(case_dirs)} case(s).\n")

processed = 0
skipped   = 0

for case_dir in sorted(case_dirs):
    case_name = os.path.basename(case_dir)
    time_avg  = os.path.join(case_dir, "time_average")
    out_dir   = os.path.join(time_avg, "average_temperature")

    win1_src = os.path.join(time_avg, "window_1_corrected_interp.csv")
    win2_src = os.path.join(time_avg, "window_2_corrected_interp.csv")

    if not os.path.isfile(win1_src) and not os.path.isfile(win2_src):
        print(f"[SKIP]  {case_name}  — no source CSVs in time_average/")
        skipped += 1
        continue

    os.makedirs(out_dir, exist_ok=True)

    for win_idx, src in enumerate([win1_src, win2_src], start=1):
        if not os.path.isfile(src):
            print(f"  [WARN] {case_name} — window_{win_idx} not found, skipping")
            continue

        temp_map = load_temperature(src)

        csv_dst  = os.path.join(out_dir, f"window_{win_idx}_temperature.csv")
        plot_dst = os.path.join(out_dir, f"window_{win_idx}_temperature.png")

        save_temperature_csv(temp_map, csv_dst)
        save_temperature_plot(temp_map, plot_dst, case_name, win_idx)

        print(f"  [OK]   {case_name} → window_{win_idx}_temperature  .csv + .png")

    processed += 1

print(f"\nDone.  Processed: {processed}   Skipped: {skipped}")
print("Outputs in each case's  time_average/average_temperature/  folder.")

Found 15 case(s).

  [OK]   cr_1400_phi_085 → window_1_temperature  .csv + .png
  [OK]   cr_1400_phi_085 → window_2_temperature  .csv + .png
  [OK]   cr_1400_phi_09 → window_1_temperature  .csv + .png
  [OK]   cr_1400_phi_09 → window_2_temperature  .csv + .png
  [OK]   cr_1400_phi_10 → window_1_temperature  .csv + .png
  [OK]   cr_1400_phi_10 → window_2_temperature  .csv + .png
  [OK]   cr_1400_phi_118 → window_1_temperature  .csv + .png
  [OK]   cr_1400_phi_118 → window_2_temperature  .csv + .png
  [OK]   cr_1400_phi_137 → window_1_temperature  .csv + .png
  [OK]   cr_1400_phi_137 → window_2_temperature  .csv + .png
  [OK]   cr_2300_phi_085 → window_1_temperature  .csv + .png
  [OK]   cr_2300_phi_085 → window_2_temperature  .csv + .png
  [OK]   cr_2300_phi_09 → window_1_temperature  .csv + .png
  [OK]   cr_2300_phi_09 → window_2_temperature  .csv + .png
  [OK]   cr_2300_phi_10 → window_1_temperature  .csv + .png
  [OK]   cr_2300_phi_10 → window_2_temperature  .csv + .png
  [OK]   cr_2